# Stage 6a: LSTM, the model that reads the story day by day

**The idea.** XGBoost looks at a *snapshot* of clues for one day. An **LSTM** (a kind of neural network) reads the **last 40 days in order**, like reading the last 40 pages of a story instead of one summary card. It carries a small "memory" from day to day and learns what patterns of calm and storm tend to come before a wiggly week.

Inside, it is a chain of tiny units, each receiving one day's clues plus the memory from the day before, and passing on an updated memory. After day 40, the memory goes through one last layer that outputs the forecast.

**Same rules as before:** the exam is the last 3 years, split into six 6-month blocks; before each block the model is re-trained using only the past, with a 10-day gap; we stop training when a practice slice (the last year of the training data) stops improving.

**Honest note:** neural networks are harder to explain and easier to overfit. They have to *earn* a place by clearly beating XGBoost, which is simpler.

Kernel: **RiskWise (.venv)**. This notebook trains on your Mac's graphics chip when it can.

In [ ]:
# Cell 1: tools and data
import sys
sys.path.append("../pipeline")
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from features import FEATURE_COLS

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("training on:", device)

tab = pd.read_parquet("../data/features.parquet")
tab["date"] = pd.to_datetime(tab["date"])
tab = tab.sort_values(["sym", "date"]).reset_index(drop=True)     # each stock's days in order, one stock after another
tab["pos"] = tab.groupby("sym").cumcount()                         # day number within each stock
print(f"{len(tab):,} rows")

## Building "the last 40 days" for every row

For a row at position `i`, its story is rows `i-39 ... i` **of the same stock**. A row is only usable once its stock has at least 40 earlier days (`pos >= 39`). The helper below gathers those 40 rows for a batch of rows at once.

In [ ]:
# Cell 2: the model and the helper that fetches 40-day stories
L = 40                                  # days of history the LSTM reads
HIDDEN = 32                             # size of its memory

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(len(FEATURE_COLS), HIDDEN, batch_first=True)
        self.out = nn.Linear(HIDDEN, 1)
    def forward(self, x):                       # x: (batch, 40 days, clues)
        memory_per_day, _ = self.rnn(x)
        return self.out(memory_per_day[:, -1]).squeeze(-1)   # forecast from the memory after day 40

offsets = torch.arange(-L + 1, 1, device=device)
def story_batch(X, y, idx):
    idx = torch.as_tensor(idx, device=device)
    return X[idx[:, None] + offsets], y[idx]            # rows i-39 ... i

print("parameters to learn:", sum(p.numel() for p in Net().parameters()))

Only about 5,000 numbers to learn, tiny compared with famous AI models. That is intentional: less to memorise.

## Walk forward, 6 blocks

For each block: standardise the clues using **training rows only**, train with early stopping, then forecast the block.

In [ ]:
# Cell 3: the walk-forward loop (about two minutes)
EXAM_START = tab["date"].max() - pd.DateOffset(years=3)
edges = [EXAM_START + pd.DateOffset(months=6 * k) for k in range(7)]
edges[-1] = tab["date"].max() + pd.Timedelta(days=1)
usable = tab["pos"] >= L - 1
every_8th = tab["pos"] % 8 == 0                  # neighbouring days overlap, so we skip most of them

blocks = []
for k in range(6):
    torch.manual_seed(k); np.random.seed(k)
    start, end = edges[k], edges[k + 1]
    cutoff = start - pd.Timedelta(days=10)
    practice_start = cutoff - pd.DateOffset(years=1)
    train_rows = tab[(tab["date"] <= cutoff) & tab["target"].notna() & usable]

    mu, sd = train_rows[FEATURE_COLS].mean().values, train_rows[FEATURE_COLS].std().values   # training rows only
    ymu, ysd = train_rows["target"].mean(), train_rows["target"].std()
    X = torch.from_numpy(np.nan_to_num(((tab[FEATURE_COLS].values - mu) / sd).astype(np.float32))).to(device)
    y_np = ((tab["target"].values - ymu) / ysd).astype(np.float32)
    y = torch.from_numpy(np.nan_to_num(y_np)).to(device)

    sel = train_rows[every_8th[train_rows.index]]
    learn_idx = sel[sel["date"] <= practice_start].index.values
    practice_idx = sel[sel["date"] > practice_start].index.values

    net = Net().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    best, best_state, patience = 9e9, None, 0
    for epoch in range(1, 13):
        net.train()
        order = np.random.permutation(learn_idx)
        for i in range(0, len(order), 1024):
            xb, yb = story_batch(X, y, order[i:i + 1024])
            opt.zero_grad(); loss = nn.functional.mse_loss(net(xb), yb); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            pred = torch.cat([net(story_batch(X, y, practice_idx[i:i + 8192])[0]) for i in range(0, len(practice_idx), 8192)])
        practice_loss = float(((pred.cpu().numpy() - y_np[practice_idx]) ** 2).mean())
        if practice_loss < best - 1e-4:
            best, patience, best_state = practice_loss, 0, {n: v.clone() for n, v in net.state_dict().items()}
        else:
            patience += 1
            if patience >= 2:                         # practice score stopped improving: stop
                break
    net.load_state_dict(best_state); net.eval()

    test = tab[(tab["date"] >= start) & (tab["date"] < end) & tab["actual_next5"].notna() & usable]
    ti = test.index.values
    with torch.no_grad():
        p = torch.cat([net(story_batch(X, y, ti[i:i + 8192])[0]) for i in range(0, len(ti), 8192)]).cpu().numpy()
    out = test[["sym", "date", "actual_next5", "naive5", "naive20"]].copy()
    out["lstm"] = np.exp(p * ysd + ymu)
    out["block"] = k + 1
    blocks.append(out)
    print(f"block {k + 1}: {start.date()} to {end.date()}  epochs used {epoch}  best practice loss {best:.4f}  exam rows {len(test):,}")
res = pd.concat(blocks)

## Proof of no peeking

Two checks: (1) every training row of the last block ended before the block began (with the 10-day gap), and (2) a row's 40-day story only contains rows of the same stock with dates up to and including that row.

In [ ]:
# Cell 4: no-peeking checks
assert train_rows["date"].max() <= edges[5] - pd.Timedelta(days=10), "training reached into the exam block!"
print("latest training date:", train_rows["date"].max().date(), "| last block starts:", edges[5].date())

i = int(test.index[len(test) // 2])
story = tab.iloc[i - L + 1: i + 1]
assert story["sym"].nunique() == 1 and story["date"].max() == tab.loc[i, "date"], "the story contains the wrong rows!"
print("story for", tab.loc[i, "sym"], "on", tab.loc[i, "date"].date(), "uses rows from", story["date"].min().date(), "to", story["date"].max().date(), "(same stock, nothing later)")

## How did it do? Against Naive, Naive-20 and XGBoost

We compare on the **same rows** (the LSTM needs 40 days of story, so a few early rows of new listings drop out of everyone's score).

In [ ]:
# Cell 5: scoreboard on the same rows
xg = pd.read_parquet("../data/preds_xgb.parquet")[["sym", "date", "xgb"]]
xg["date"] = pd.to_datetime(xg["date"])
both = res.merge(xg, on=["sym", "date"], how="inner")

def share(df, col):
    g = df.assign(miss=(df["actual_next5"] - df[col]).abs()).groupby("sym")
    return g["miss"].mean() / g["actual_next5"].median()

table = pd.DataFrame({"Naive": share(both, "naive5"), "Naive-20": share(both, "naive20"),
                      "XGBoost": share(both, "xgb"), "LSTM": share(both, "lstm")})
print("Average miss as share of typical wiggle (mean over", len(table), "stocks, lower is better):")
print(table.mean().round(3).to_string())
print("\nLSTM beats Naive on", round((table["LSTM"] < table["Naive"]).mean() * 100, 1), "% of stocks")
print("LSTM beats XGBoost on", round((table["LSTM"] < table["XGBoost"]).mean() * 100, 1), "% of stocks")

In [ ]:
# Cell 6: block by block, is the LSTM's result stable?
rows = []
for k, part in both.groupby("block"):
    rows.append({"block": k, **{name: share(part, col).mean() for name, col in
                 [("Naive", "naive5"), ("Naive-20", "naive20"), ("XGBoost", "xgb"), ("LSTM", "lstm")]}})
pd.DataFrame(rows).set_index("block").round(3)

In [ ]:
# Cell 7: save the LSTM's exam guesses for the final scoreboard
res[["sym", "date", "lstm", "block"]].to_parquet("../data/preds_lstm.parquet", index=False)
print("saved ../data/preds_lstm.parquet:", f"{len(res):,}", "rows")

## Reading the results honestly

If the LSTM is about level with XGBoost (not clearly better), the sensible conclusion is: **keep the simpler model**. A neural network that does not clearly win is more to maintain, harder to explain on the How it works page, and slower. The scoreboard notebook decides what ships.

## Your turn

1. Change `L = 40` to `L = 20` and then `L = 80` (Cell 2) and re-run Cells 3 to 5. Does a longer story help?
2. Change `HIDDEN = 32` to `HIDDEN = 8`, then `128`. Does a bigger memory help, or just overfit?
3. Explain back: **why do we standardise the clues using the training rows only?**